# 직접 만져 보는 대화 기억과 사람의 승인

**먼저 `USE = "fake"`로 모든 셀을 위에서 아래로 실행하세요.** 가짜 모델은 GPU도 API도 쓰지 않습니다. Gemma 셀은 연결 객체만 만들고, Solar 셀도 실제 질문을 보내지 않습니다. 처음에는 Solar 셀을 건너뛰어도 됩니다. 이 실습의 주문은 노트북 안의 장난감 데이터입니다.

배울 것은 두 가지입니다. 모델의 **대화 맥락(context)** 은 이번 요청에 함께 보낸 대화 목록입니다. **HITL(human-in-the-loop)** 은 프로그램이 중요한 작업을 실행하기 전에 멈추고 사람의 결정을 받는 방식입니다. “취소할까요?”라는 문장만 출력하는 것과 실제 실행을 막아 두는 것은 다릅니다.

| 순서 | 해 볼 일 |
|---|---|
| 준비 | 환경 확인 → Gemma / Solar / 가짜 모델 준비 → 하나 선택 |
| 1. 대화 기억 | 메시지 목록, 같은 대화와 새 대화, 저장된 기록 비교 |
| 2. 사람의 승인 | 주문 조회와 취소, 승인 / 거절 / 대신 답하기 / 수정 비교 |
| 3. shopagent와 연결 | 거절 문구를 한국어로 바꾸고 실제 동의 변환 함수 호출 |
| 4. 직접 바꿔 볼 것 | 일곱 가지 실험 |

작은 용어장입니다. 코드에서 쓰기 직전에도 다시 설명합니다.

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 메시지(message) | 누가 무엇을 말했는지를 담은 항목. 사람, 모델, 시스템 지시, 도구 결과로 나뉩니다. |
| 상태(state) | 프로그램이 다음 단계로 넘기는 자료 묶음. 여기서는 주로 `messages`라는 대화 목록입니다. |
| 그래프(graph), 노드(node), 연결(edge) | 실행 순서, 그 안의 작업 한 단계, 단계 사이 이동 경로입니다. |
| 체크포인터(checkpointer), 스냅샷(snapshot) | 상태를 단계마다 저장하는 장치와 그렇게 저장된 상태 한 벌입니다. |
| `thread_id` | 저장된 여러 대화를 구별하는 이름표입니다. 사용자 인증 기능은 아닙니다. |
| 도구(tool), 도구 호출(tool call) | 파이썬 함수와 모델이 그 함수의 이름·입력값을 지정한 실행 요청입니다. 요청 자체는 아직 실행이 아닙니다. |
| 미들웨어(middleware) | 모델이나 도구의 실행 사이에 끼어들어 동작을 검사하거나 바꾸는 코드입니다. |
| 중단(interrupt), 재개(resume) | 실행할 일을 남겨 두고 멈추는 것, 사람의 결정과 함께 이어 가는 것입니다. |

노트북의 **커널(kernel)** 은 셀들이 공유하는 파이썬 실행 프로세스입니다. 아래 메모리 저장소는 그 프로세스 안에만 있으므로 커널을 재시작하면 사라집니다. 셀을 다시 실행하며 결과가 섞이면, 커널을 재시작하고 위에서부터 실행하세요.

## 준비: 프로젝트와 출력 도우미

작업 폴더와 `sys.path`(파이썬이 모듈을 찾는 경로)를 프로젝트로 맞춥니다. `import`는 사용할 코드를 불러오는 명령입니다. LangChain은 모델·도구를 연결하고, LangGraph는 상태를 단계 사이에 넘기고 저장합니다. 버전은 지금 커널에 설치된 값을 출력합니다.

`HumanMessage`는 사람의 말, `AIMessage`는 모델의 말, `SystemMessage`는 모델에게 주는 지시, `ToolMessage`는 도구 결과입니다. `show()`는 각 메시지를 한 줄로 보여 줍니다. 도구 응답의 `name`은 도구 이름, `status`는 결과 표식입니다. `success`라고 해서 주문 취소 같은 업무가 성공했다는 뜻은 아닙니다. 내용도 읽어야 합니다.

`Annotated`는 함수 입력값에 설명을 붙일 때, `uuid4()`는 겹치지 않는 식별자를 만들 때 씁니다. 객체는 파이썬에서 사용할 값과 기능을 묶어 둔 것입니다. 아래 임포트나 객체 생성은 모델에 질문을 보내지 않습니다.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from importlib.metadata import version
from typing import Annotated
from uuid import uuid4

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package}: {version(package)}")

def show(messages):
    for message in messages:
        line = f"{message.type}: {message.content}"
        if isinstance(message, AIMessage) and message.tool_calls:
            line += " | tool_calls=" + json.dumps(message.tool_calls, ensure_ascii=False)
        if isinstance(message, ToolMessage):
            line += f" | name={message.name} status={message.status}"
        print(line.replace("\n", " ↵ "))

def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma: shopagent와 같은 로컬 모델

**로컬(local)** 은 내 컴퓨터에서 실행한다는 뜻입니다. `LocalLLM(...).client`는 프로젝트가 쓰는 모델 연결 객체입니다. 객체를 만드는 것과 실제로 질문하는 것은 다릅니다. 이 셀은 연결 객체를 만든 뒤 최대 2초 동안 `/health`로 서버 상태를 확인하고, 상태 코드나 서버가 꺼져 있다는 안내를 출력합니다. 모델에 질문을 보내지는 않습니다.

가짜 모델 실습에는 서버가 필요 없습니다. 나중에 Gemma 4를 선택할 때만 **별도 터미널**에서 shopagent의 `g12-native`와 같은 명령으로 서버를 실행하세요. 노트북은 서버를 시작하지 않습니다.

```bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
```

끄려면 서버를 실행한 터미널에서 **Ctrl+C**를 누르세요. `pgrep -a llama-server`로 남아 있는 서버가 있는지 확인할 수 있습니다. `127.0.0.1:18380`은 이 컴퓨터의 서버 주소입니다. 객체가 만들어졌다고 서버까지 준비된 것은 아닙니다.

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
print("Gemma 연결 객체 준비 완료. 모델 호출은 하지 않았습니다.")
try:
    with httpx.Client(trust_env=False) as health_client:
        health_response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    print(f"llama-server 켜짐 (상태 코드: {health_response.status_code})")
except httpx.HTTPError:
    print("llama-server 꺼짐: 위의 별도 터미널 명령으로 서버를 실행하세요.")

### Solar: shopagent와 같은 Upstage 설정

**API**는 외부 서비스에 요청하는 통로이고, **API 키**는 그 서비스를 사용할 비밀 인증값입니다. `CONFIGS["solar-pro4"]`는 프로젝트의 Solar 설정입니다. **객체 생성은 비용이 들지 않지만 실제 모델 호출에는 비용이 듭니다.** 처음에는 이 셀을 건너뛰고 가짜 모델을 쓰세요.

키 파일은 파이썬으로 열지 않습니다. 아래 코드는 bash 셸의 `source`에 해당하는 `.` 명령으로만 `upstage.env`를 불러옵니다. `set -a`는 불러온 값을 환경 변수(프로세스가 보관하는 설정값)로 내보냅니다. 출력은 내부에서 받아 키가 비어 있지 않을 때만 `os.environ["UPSTAGE_API_KEY"]`에 넣고, 로드 여부만 True/False로 표시합니다. 키가 없으면 `solar_llm = None`입니다.

**`print(os.environ)`을 하지 마세요.** 셀 출력은 노트북 파일에 저장될 수 있습니다. 키나 셸의 출력·오류 내용을 출력하지 마세요. 이 파일은 출력이 비어 있는 상태로 제공합니다.

In [ ]:
from shop.servers import CONFIGS

def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO, capture_output=True, text=True, timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())

solar_loaded = load_solar_key()
print("Upstage 키 loaded:", solar_loaded)
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
if solar_llm is None:
    print("키가 없어 Solar를 준비하지 못했습니다. USE = 'fake'로 진행하세요.")

### 가짜 모델: 정해 둔 규칙만 실행

**LLM**은 언어 모델을 부르는 이름입니다. 이 가짜 모델은 학습된 모델 대신 작은 파이썬 규칙으로 답합니다. `BaseChatModel`(대화 모델의 공통 틀)을 상속(기존 틀의 기능을 물려받는 것)하고, `bind_tools`(사용 가능한 도구를 모델에 연결하는 메서드, 즉 객체에 속한 함수)는 자기 자신을 반환합니다. `ChatResult`와 `ChatGeneration`은 답변을 LangChain이 받는 모양으로 감쌉니다.

마지막 메시지가 도구 결과면 그 내용을 읽었다고 답합니다. 그 외에는 마지막 사람 말에서 공백과 문장부호로 나눈 `O001` 같은 주문 번호를 찾습니다. 번호와 “취소”가 있으면 `cancel_order`, 번호만 있으면 `get_order` 호출을 요청합니다. 아직 함수를 실행하지는 않습니다.

나머지는 시스템 지시를 뺀 메시지 수와 첫 메시지를 답합니다. 모델 객체 안에 대화를 저장하지 않으므로, **어떤 목록을 보냈는지**가 답에 그대로 드러납니다.

In [ ]:
class PlaygroundChatModel(BaseChatModel):
    @property
    def _llm_type(self):
        return "playground-scripted"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if messages and isinstance(messages[-1], ToolMessage):
            reply = AIMessage(content=f"(가짜 모델) 도구 결과를 받았어요: {messages[-1].content}")
        else:
            humans = [m for m in messages if isinstance(m, HumanMessage)]
            human_text = str(humans[-1].content) if humans else ""
            words = human_text
            for separator in ",.!?;:()[]{}\n":
                words = words.replace(separator, " ")
            order_id = next((word for word in words.split()
                             if word.startswith("O") and word[1:].isdigit()), None)
            if order_id:
                name = "cancel_order" if "취소" in human_text else "get_order"
                reply = AIMessage(content="", tool_calls=[{
                    "name": name, "args": {"order_id": order_id},
                    "id": "call-" + uuid4().hex[:8], "type": "tool_call",
                }])
            else:
                visible = [m for m in messages if not isinstance(m, SystemMessage)]
                first = visible[0].content if visible else "(없음)"
                reply = AIMessage(content=f"(가짜 모델) 메시지 {len(visible)}개를 봤어요. 첫 메시지: {first}")
        return ChatResult(generations=[ChatGeneration(message=reply)])

fake_llm = PlaygroundChatModel()
print("가짜 모델 준비 완료: GPU·API 없이 실행합니다.")

### 이번 실습에서 쓸 모델 하나 선택

`invoke`는 모델이나 그래프를 한 번 실행하는 메서드입니다. 아래 선택 셀은 아직 `invoke`하지 않습니다. 이후 셀은 모두 `llm`을 쓰므로, 선택을 바꾼 뒤에는 **이 셀부터 아래를 다시 실행**하세요. 가짜 모델의 숫자와 결과는 재현할 수 있지만 실제 모델의 답변과 도구 선택은 달라질 수 있습니다.

Solar 셀을 건너뛰었거나 키가 없으면 Solar 선택 시 안내와 함께 멈춥니다. Gemma는 서버를 직접 준비해야 하며, 준비하지 않았다면 다음 호출에서 연결 오류가 납니다.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"
models = {"fake": fake_llm, "gemma": globals().get("gemma_llm"), "solar": globals().get("solar_llm")}
assert USE in models, "USE는 fake, gemma, solar 중 하나여야 합니다."
llm = models[USE]
assert llm is not None, "선택한 모델이 준비되지 않았습니다. 해당 준비 셀을 실행하거나 USE = 'fake'로 바꾸세요."
print("선택:", USE)
if USE == "gemma":
    print("위 Gemma 셀의 서버 상태 검사 결과를 확인하세요. 서버가 꺼져 있다면 위 터미널 명령으로 실행한 뒤 진행하세요.")

## 1. 대화 기억: 모델에게 무엇을 다시 보내는가

먼저 그래프 없이 모델에 메시지 **리스트(list, 순서가 있는 목록)** 를 직접 보냅니다. 첫 질문 뒤에 새 질문만 보내는 경우와 이전 질문·답변까지 함께 보내는 경우를 비교하세요. 가짜 모델은 각각 메시지 1개, 1개, 3개를 봤다고 답합니다. 기억처럼 보이는 것은 이전 메시지를 다시 넣어 준 결과입니다.

In [ ]:
first_question = HumanMessage(content="제 별명은 도토리예요.")
followup = HumanMessage(content="제 별명을 기억하나요?")
first_answer = llm.invoke([first_question])
print("첫 질문:")
show([first_answer])
print("새 질문만 전송:")
show([llm.invoke([followup])])
print("이전 대화도 함께 전송:")
show([llm.invoke([first_question, first_answer, followup])])

### 메시지 목록을 대신 보관하는 그래프

`MessagesState`는 `messages` 목록이 있는 상태의 기본 틀입니다. 새 메시지를 반환하면 기존 목록에 합칩니다. 같은 메시지 ID를 다시 주면 해당 항목을 바꿀 수 있지만, 여기서는 새 메시지만 추가합니다.

`StateGraph`로 **시작(START) → 모델 노드 → 끝(END)** 순서를 만들고, `compile`로 실행할 수 있게 만듭니다. 노드는 받은 상태에서 메시지 전체를 모델에 보내고 새 답변 하나를 반환합니다. `InMemorySaver`는 스냅샷을 현재 파이썬 메모리에 저장하는 체크포인터입니다. 모델 내부를 바꾸는 장치는 아닙니다.

In [ ]:
def model_node(state):
    return {"messages": [llm.invoke(state["messages"])]}

memory_builder = StateGraph(MessagesState)
memory_builder.add_node("model", model_node)
memory_builder.add_edge(START, "model")
memory_builder.add_edge("model", END)
memory_saver = InMemorySaver()
memory_graph = memory_builder.compile(checkpointer=memory_saver)
print("시작 → 모델 → 끝, 메모리 저장소 연결 완료")

### 같은 이름표: 이전 대화를 이어 받기

`config`는 실행 설정입니다. 그 안의 `configurable.thread_id`로 저장된 대화를 고릅니다. `thread_config()`는 중복되지 않는 이름표를 하나 만듭니다. 아래 두 호출은 **동일한 `config_a`를 재사용**합니다.

호출할 때는 새 사람 말만 넣습니다. 그래프가 이전 기록을 합치고 답변을 추가하므로 전체 메시지는 2개에서 4개로 늘어납니다. 두 번째 모델 호출 직전에는 사람·모델·사람, 총 3개가 들어 있습니다.

In [ ]:
config_a = thread_config("memory-a")
turn_a1 = memory_graph.invoke({"messages": [HumanMessage(content="제 별명은 도토리예요.")]}, config_a)
print("첫 호출 뒤 메시지 수:", len(turn_a1["messages"]))
show(turn_a1["messages"])
turn_a2 = memory_graph.invoke({"messages": [HumanMessage(content="제 별명을 기억하나요?")]}, config_a)
print("두 번째 호출 뒤 메시지 수:", len(turn_a2["messages"]))
show(turn_a2["messages"])

### 새 이름표: 같은 모델인데 이전 대화가 없다

같은 그래프와 같은 저장소를 쓰더라도 다른 `thread_id`이면 다른 대화입니다. 아래 결과에는 새 질문과 새 답변, 2개만 있습니다. 가짜 모델은 첫 메시지가 “제 별명을 기억하나요?”라고 말합니다.

In [ ]:
config_b = thread_config("memory-b")
turn_b = memory_graph.invoke({"messages": [HumanMessage(content="제 별명을 기억하나요?")]}, config_b)
print("새 대화의 메시지 수:", len(turn_b["messages"]))
show(turn_b["messages"])

### 저장된 상태 열어 보기

`get_state(config)`는 해당 대화의 최신 스냅샷을 읽습니다. `values["messages"]`는 모델에 전달하는 목록의 실체입니다. 여기서는 모델 노드가 이 목록을 그대로 넘깁니다. **호출이 끝난 뒤의 상태에는 방금 생성한 답변도 포함**되므로 마지막 모델 요청의 입력보다 하나 많습니다. 다른 지시를 덧붙이는 에이전트와는 이 점을 구분하세요.

`next`는 다음에 실행할 노드들의 이름입니다. 완료했으면 빈 튜플(tuple, 순서가 있고 내용을 바꿀 수 없는 자료 묶음) `()`입니다. `checkpoint_id`는 스냅샷 한 벌의 식별자이며, 대화 이름표인 `thread_id`와 다릅니다.

In [ ]:
snapshot_a = memory_graph.get_state(config_a)
show(snapshot_a.values["messages"])
print("next:", snapshot_a.next)
print("checkpoint_id:", snapshot_a.config["configurable"]["checkpoint_id"])

### 저장된 모든 스냅샷 보기

`get_state_history`는 최신 것부터 과거 순서로 저장 기록을 꺼냅니다. `step`은 그래프의 실행 단계 번호입니다. 사용자가 말한 횟수와 일치하지 않습니다. 입력을 받은 시점, 모델이 끝난 시점 등이 따로 저장되므로 메시지 수가 같은 스냅샷도 보입니다. 아래 도우미는 기록을 하나도 생략하지 않고 단계·메시지 수·다음 작업을 출력합니다.

In [ ]:
def show_history(graph, config):
    for saved in graph.get_state_history(config):
        print("step:", saved.metadata.get("step"),
              "| messages:", len(saved.values.get("messages", [])),
              "| next:", saved.next)

show_history(memory_graph, config_a)

### 체크포인터를 빼면: 기억도, 저장된 중단에서 재개도 없다

같은 실행 순서를 체크포인터 없이 만들면 같은 이름표로 불러도 이전 메시지를 합치지 못합니다. 두 호출 모두 결과는 2개입니다. `get_state`도 저장소가 없다는 오류를 냅니다.

`interrupt()`는 중단 사유를 내보내고, `Command(resume=...)`는 사람의 답을 전달하는 재개 명령입니다. 저장소 없이도 **중단 신호 자체는 나올 수 있지만, 그 지점에서 이어 갈 상태가 없어 재개할 수 없습니다.** 아래 작은 중단 전용 그래프로 이 차이를 확인합니다. 예상한 두 오류는 잡아서 짧게 출력합니다.

`InMemorySaver`가 있어도 디스크에는 저장하지 않습니다. 새 저장소로 바꾸거나 커널을 재시작하면 이전 기록은 사라집니다.

In [ ]:
plain_graph = memory_builder.compile()
for text in ("제 별명은 도토리예요.", "제 별명을 기억하나요?"):
    result = plain_graph.invoke({"messages": [HumanMessage(content=text)]}, config_a)
    print("저장소 없는 호출의 메시지 수:", len(result["messages"]))
    show(result["messages"])
try:
    plain_graph.get_state(config_a)
except ValueError as error:
    print("상태 조회 불가:", str(error))

def ask_to_continue(state):
    interrupt("계속할까요?")
    return {}

pause_builder = StateGraph(MessagesState)
pause_builder.add_node("ask", ask_to_continue)
pause_builder.add_edge(START, "ask")
pause_builder.add_edge("ask", END)
no_saver_pause = pause_builder.compile()
pause_signal = no_saver_pause.invoke({"messages": []}, config_a)
print("중단 신호는 나옴:", bool(pause_signal.get("__interrupt__")))
try:
    no_saver_pause.invoke(Command(resume={pause_signal["__interrupt__"][0].id: "계속"}), config_a)
except RuntimeError as error:
    print("재개 불가:", str(error))

## 2. 사람의 승인: 조회는 바로, 취소는 멈춘 뒤 실행

장난감 주문 세 개를 만듭니다. `orders`는 주문 번호로 상태를 찾는 딕셔너리(이름과 값의 짝을 모은 자료)입니다. 실제 쇼핑몰이나 shopagent 데이터에는 연결하지 않습니다. `reset_orders()`는 실험을 반복하기 위한 초기화 함수입니다. **주문 데이터와 대화 저장소는 서로 별개**입니다. 새 대화를 만들어도 주문이 자동으로 초기화되지는 않습니다.

`@tool`은 함수를 모델이 요청할 수 있는 도구로 등록합니다. 함수 아래의 한국어 문자열은 도구 설명이고, `Annotated`는 `order_id` 입력값을 설명합니다. 조회는 읽기만 합니다. 취소는 실행하면 주문을 바꾸며 되돌리는 도구를 제공하지 않습니다. 배송중이면 거부합니다. `tool_runs`는 실제로 실행된 함수 이름과 입력값을 남겨, 요청과 실행을 구분하게 합니다.

In [ ]:
orders = {}
tool_runs = []

def reset_orders():
    orders.clear()
    orders.update({
        "O001": {"status": "결제완료"},
        "O002": {"status": "상품준비중"},
        "O003": {"status": "배송중"},
    })
    tool_runs.clear()

@tool
def get_order(order_id: Annotated[str, "조회할 주문 번호. 예: O001"]) -> dict:
    """주문 상태를 조회합니다. 주문을 변경하지 않습니다."""
    tool_runs.append(("get_order", order_id))
    return {"order_id": order_id, "order": orders.get(order_id, {}).copy()}

@tool
def cancel_order(order_id: Annotated[str, "취소할 주문 번호. 예: O001"]) -> dict:
    """주문을 취소합니다. 되돌리는 도구는 없으며 배송중인 주문은 취소하지 않습니다."""
    tool_runs.append(("cancel_order", order_id))
    if order_id not in orders:
        return {"ok": False, "reason": "없는 주문입니다."}
    if orders[order_id]["status"] == "배송중":
        return {"ok": False, "reason": "배송중이라 취소하지 않았습니다."}
    orders[order_id]["status"] = "취소완료"
    return {"ok": True, "order_id": order_id, "status": "취소완료"}

reset_orders()
print(orders)

### 실행 직전에 멈추도록 에이전트 만들기

**에이전트(agent)** 는 모델의 답이 도구 호출이면 그 도구를 실행하고, 결과를 다시 모델에 보내는 프로그램입니다. `create_agent`가 이 반복 흐름을 그래프로 만듭니다. `system_prompt`는 모델에게 주는 공통 지시입니다. 이 지시는 에이전트가 모델 호출에 덧붙이며, 저장된 대화 목록 자체와는 구별됩니다.

`HumanInTheLoopMiddleware`는 지정한 도구가 실행되기 직전에 멈추게 하는 미들웨어입니다. `interrupt_on`은 멈출 도구, `allowed_decisions`는 사람이 고를 수 있는 결정입니다. `description` 함수는 사람이 읽을 확인 카드의 문구를 만듭니다. 재개할 때도 호출될 수 있으므로 주문 변경 같은 작업은 넣지 않습니다.

`interrupt_mode="per_call"`은 **도구 호출 한 건마다 별도의 중단**을 만듭니다. 조회는 목록에 넣지 않아 바로 실행됩니다. 이 노트북은 approve(그대로 실행), reject(거절), respond(실행 대신 사람의 답변 전달), edit(입력 수정 후 실행)를 모두 허용합니다. **shopagent v2는 approve/reject/respond만 허용**합니다. 이 함수로 만든 에이전트마다 새 `InMemorySaver`를 붙입니다.

In [ ]:
order_tools = [get_order, cancel_order]
order_prompt = (
    "한국어 주문 실습 도우미입니다. 주문 조회는 get_order를 호출하세요. "
    "주문 취소 요청은 cancel_order를 직접 호출하세요. 승인은 실행 직전 프로그램이 받습니다. "
    "확인 질문만 문장으로 답하지 마세요. 도구 결과 없이 취소 완료라고 말하지 마세요."
)

def confirmation_card(call, state, runtime):
    order_id = call["args"]["order_id"]
    return f"확인: {order_id}에 {call['name']} 실행 / 현재 상태: {orders.get(order_id)}. 진행할까요?"

def make_order_agent(allowed_decisions=None, *, interrupt_reads=False,
                     middleware_class=HumanInTheLoopMiddleware):
    allowed = ["approve", "reject", "respond", "edit"] if allowed_decisions is None else allowed_decisions
    interrupt_on = {"cancel_order": {"allowed_decisions": allowed, "description": confirmation_card}}
    if interrupt_reads:
        interrupt_on["get_order"] = {"allowed_decisions": allowed, "description": confirmation_card}
    return create_agent(
        llm, order_tools, system_prompt=order_prompt,
        middleware=[middleware_class(interrupt_on=interrupt_on, interrupt_mode="per_call")],
        checkpointer=InMemorySaver(),
    )

order_agent = make_order_agent()
print("취소 도구 실행 전에 사람의 결정을 받습니다.")

### 조회: 중단 없이 결과까지

`get_order`는 중단 대상이 아닙니다. 가짜 모델에서는 `사람 → 모델의 도구 호출 → 도구 결과 → 모델의 설명` 순서가 보이고, `interrupts`는 비어 있습니다. 주문 상태도 바뀌지 않습니다. 실제 모델을 선택했다면 도구 호출이 실제로 나왔는지도 함께 보세요.

In [ ]:
reset_orders()
lookup_config = thread_config("lookup")
lookup_result = order_agent.invoke({"messages": [HumanMessage(content="주문 O001 상태를 알려 주세요.")]}, lookup_config)
show(lookup_result["messages"])
print("interrupts:", order_agent.get_state(lookup_config).interrupts)
print("주문:", orders)
print("실행된 도구:", tool_runs)

### 취소 요청: 요청은 생겼지만 아직 실행되지 않았다

아래 도우미는 매번 주문을 초기화하고 새 대화에서 취소를 요청합니다. 모델이 `cancel_order` 호출을 내면 중단됩니다. `interrupt.id`는 **이 중단에 답할 때 사용할 식별자**, `interrupt.value`는 **호출 이름·입력값·확인 문구가 담긴 자료(payload)** 입니다. `next`에는 아직 할 일이 남아 있습니다. `tool_runs`가 비어 있고 O001이 결제완료인지 확인하세요.

**실제 모델이 도구 대신 “취소할까요?”라는 글만 답하면 중단이 생기지 않습니다.** shopagent에서 확인 단계 실패로 세는 경우입니다. 아래는 그 경우를 표시하고 재개 예시를 건너뜁니다. 문장으로 확인하는 것만으로는 이 실행 차단 장치를 거치지 않습니다.

In [ ]:
def start_cancel_case(label, agent=None, text="주문 O001 취소해 주세요."):
    agent = order_agent if agent is None else agent
    reset_orders()
    config = thread_config(label)
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, config)
    show(result["messages"])
    snapshot = agent.get_state(config)
    pending_interrupt = snapshot.interrupts[0] if snapshot.interrupts else None
    if pending_interrupt is not None:
        print("interrupt.id:", pending_interrupt.id)
        print("interrupt.value:", json.dumps(pending_interrupt.value, ensure_ascii=False, indent=2))
    else:
        print("중단 없음: 취소 도구 호출이 나왔는지 위 메시지를 확인하세요.")
        if not any(isinstance(m, AIMessage) and any(c["name"] == "cancel_order" for c in m.tool_calls)
                   for m in result["messages"]):
            print("확인 단계 실패: 모델이 취소 도구를 호출하지 않았습니다. 글로만 확인해도 해당합니다.")
    print("next:", snapshot.next)
    print("주문:", orders)
    print("실행된 도구:", tool_runs)
    return config, pending_interrupt, result

approve_config, approve_interrupt, paused_result = start_cancel_case("approve")

### 멈춰 있는 동안의 기록도 읽을 수 있다

사람의 결정 전에도 저장소에는 모델의 도구 요청과 이어서 실행할 일이 남아 있습니다. 앞에서 만든 `show_history`로 모든 스냅샷을 봅니다. 최신 기록의 `next`에 `tools`가 보이면 도구 단계가 끝나지 않은 것입니다. 중단이 생기지 않은 실제 모델의 경우에는 완료된 기록이 보일 수 있습니다.

In [ ]:
show_history(order_agent, approve_config)

### approve: 그대로 실행해서 이어 가기

`decision`은 사람이 고른 결정을 담은 딕셔너리입니다. `Command(resume={중단ID: 결정})`으로 **어느 중단에 대한 답인지** 지정합니다. `per_call`에서는 결정 한 개를 바로 연결하며, 여러 결정을 감싸는 `decisions` 목록을 넣지 않습니다. 처음 요청과 **같은 에이전트와 같은 대화 설정**을 써야 합니다.

`approve`이면 도구가 실행됩니다. 가짜 모델에서는 O001이 취소완료가 되고, 도구 응답은 `name=cancel_order status=success`입니다. 아래 도우미는 나머지 결정에서도 메시지·남은 중단·주문·실제 실행 기록을 똑같이 보여 줍니다.

In [ ]:
def resume_case(agent, config, pending_interrupt, decision):
    if pending_interrupt is None:
        print("재개할 중단이 없어 생략합니다. 가짜 모델로 먼저 확인하세요.")
        return None
    result = agent.invoke(Command(resume={pending_interrupt.id: decision}), config)
    show(result["messages"])
    print("남은 중단:", agent.get_state(config).interrupts)
    print("주문:", orders)
    print("실행된 도구:", tool_runs)
    return result

decision = {"type": "approve"}
approved = resume_case(order_agent, approve_config, approve_interrupt, decision)

### reject: 거절 이유를 전달하고 도구는 건너뛰기

승인 예시와 섞이지 않도록 주문을 초기화하고 새 대화에서 다시 중단시킵니다. `reject`의 `message`는 거절 이유입니다. 기본 미들웨어는 `User rejected the tool call ... with reason: ...` 같은 **영어 문구에 한국어 거절 이유를 붙인** `ToolMessage`를 만듭니다. `status="error"`이지만 파이썬 실행 오류가 난 것이 아니라 사람이 거절했다는 표식입니다.

가짜 모델에서는 O001이 결제완료로 남고 취소 도구 실행 기록도 없습니다. 뒤에서 이 영어 부분을 한국어로 바꿉니다.

In [ ]:
reject_config, reject_interrupt, _ = start_cancel_case("reject")
rejected = resume_case(order_agent, reject_config, reject_interrupt,
                       {"type": "reject", "message": "지금은 취소하지 말아 주세요."})

### respond: 실행 대신 사람의 문장을 도구 결과로 주기

`respond`는 도구를 실행하지 않고 `message`를 그대로 도구 결과로 돌려줍니다. `status="success"`는 대체 답변이 전달됐다는 표식이며 **취소 성공이라는 뜻이 아닙니다.** 가짜 모델에서 O001과 실행 기록은 거절 예시처럼 바뀌지 않습니다. shopagent는 고객 답변이 불분명한 경우에도 이 결정을 씁니다.

In [ ]:
respond_config, respond_interrupt, _ = start_cancel_case("respond")
responded = resume_case(order_agent, respond_config, respond_interrupt,
                        {"type": "respond", "message": "아직 결정하지 못했습니다. 취소를 실행하지 마세요."})

### edit: 입력을 바꿔 실행하기 — 비교용 기능

`edited_action`은 사람이 수정한 도구 이름과 입력값입니다. 같은 취소 도구의 입력을 O001에서 **O002**로 바꿉니다. 현재 `per_call`의 수정 결정은 도구 이름을 유지하고 입력 형식도 검사합니다.

가짜 모델에서 O001은 결제완료로 남고 O002가 취소완료가 됩니다. 모델의 원래 요청에는 O001이 남아 있고, 도구 응답에는 실제로 O002를 실행했다는 안내가 붙습니다. 실행 기록도 비교하세요. **shopagent v2는 edit를 허용하지 않습니다.** 여기서는 라이브러리 기능을 비교하기 위해 넣었습니다.

In [ ]:
edit_config, edit_interrupt, _ = start_cancel_case("edit")
edited = resume_case(order_agent, edit_config, edit_interrupt, {
    "type": "edit",
    "edited_action": {"name": "cancel_order", "args": {"order_id": "O002"}},
})

## 3. shopagent와 연결하기

### 거절 결과만 한국어로 바꾸는 미들웨어

아래 `RejectMessageMiddleware`는 기본 미들웨어를 상속해 `_resolve_per_call`(도구 한 건의 결정을 처리하는 내부 메서드)을 바꿉니다. 먼저 부모의 처리를 거친 다음, `status="error"`인 도구 메시지만 복사해 내용을 한국어로 바꿉니다. 도구 이름·호출 ID·상태는 유지합니다. 승인·수정의 반환값은 실행할 요청이고, `respond`는 성공 표식이므로 그대로 둡니다.

이것은 `shop/graph_v2.py`의 `ConsentRejectMessageMiddleware`와 같은 방식의 작은 예시입니다. 그 코드도 거절을 한국어로 바꾸지만, 실제 프로젝트의 문구와 제안 관리 전체를 복사하지는 않습니다. 내부 메서드를 사용하므로 LangChain 버전을 바꾸면 다시 확인해야 합니다.

In [ ]:
class RejectMessageMiddleware(HumanInTheLoopMiddleware):
    def _resolve_per_call(self, request):
        result = super()._resolve_per_call(request)
        if isinstance(result, ToolMessage) and result.status == "error":
            call = request.tool_call
            return result.model_copy(update={"content": (
                f"실행하지 않았습니다. 고객이 {call['name']} "
                f"(주문 {call['args']['order_id']})을 승인하지 않았습니다. 완료로 안내하지 마세요."
            )})
        return result

reject_message_agent = make_order_agent(middleware_class=RejectMessageMiddleware)
reject_message_config, reject_message_interrupt, _ = start_cancel_case("reject-message", agent=reject_message_agent)
reject_message_rejected = resume_case(reject_message_agent, reject_message_config, reject_message_interrupt,
                              {"type": "reject", "message": "취소하지 않을게요."})

### 실제 프로젝트 함수: 판정 결과를 재개 결정으로 바꾸기

**verdict(판정값)** 은 고객의 말에 대한 분류 결과입니다. 여기서는 모델로 판정하지 않고 네 값을 직접 넣어 봅니다. `consent_resume_decision`은 프로젝트의 실제 함수를 불러옵니다. 이 함수 자체는 모델·API를 호출하지 않습니다.

`yes`는 approve, `no`와 `changed`(요청 변경)는 기존 제안을 reject, `unclear`(불분명)는 respond로 바뀝니다. 변경 요청은 기존 작업을 수정 실행하는 edit가 아니라 **기존 제안 거절**입니다. 불분명한 답변에는 “실행되지 않았다”는 한국어 안내가 붙습니다. 이 셀은 결정 자료를 만드는 것까지만 하며 주문을 실행하지 않습니다.

In [ ]:
from shop.graph_v2 import consent_resume_decision

for verdict, customer_text in (
    ("yes", "네, 진행해 주세요."),
    ("no", "취소하지 않을게요."),
    ("changed", "다른 주문을 먼저 보고 싶어요."),
    ("unclear", "잠깐 생각해 볼게요."),
):
    print(verdict, "→", consent_resume_decision(verdict, customer_text))

### 노트북에서 본 것과 shopagent 코드의 연결

| 노트북에서 본 개념 | 프로젝트 위치와 역할 |
|---|---|
| `LocalLLM(...).client` | [shop/agents.py](../shop/agents.py)의 `LocalLLM`: 같은 Gemma 연결 방식과 원격 API 연결 방식 |
| Solar 설정과 Gemma 실행 설정 | [shop/servers.py](../shop/servers.py)의 `CONFIGS`, `make`, `resident`: `solar-pro4`, `g12-native` |
| 에이전트와 확인 카드 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_agent`, 내부 `describe`: 실행 전 제안·확인 문구를 만들고 `per_call`로 중단 |
| 한국어 거절 결과 | 같은 파일의 `ConsentRejectMessageMiddleware._resolve_per_call`: 거절 도구 메시지의 내용 교체 |
| 판정값 → 재개 결정 | 같은 파일의 `judge_consent`, `consent_resume_decision`: 판정과 결정 변환을 분리 |
| 메모리 저장소와 대화 이름표 | 같은 파일의 `ScenarioGraph.__init__`, `build_graph`: 바깥 그래프에 `InMemorySaver`와 시나리오별 `thread_id`를 붙임. 담당 에이전트는 이 저장 흐름 안에서 실행 |
| 중단을 찾아 재개 | 같은 파일의 `ScenarioGraph.run_turn`, `resume_turn`: 중단 ID에 approve/reject/respond를 연결한 `Command` 실행 |
| 거절 후 같은 작업 재호출 관리 | 같은 파일의 `TurnMiddleware`, `declined_actions`: 거절 이력을 보고 다시 실행하려는 시도를 관리. 이 작은 노트북은 이 정책을 구현하지 않음 |

노트북은 모델의 도구 선택 → 실행 직전 중단 → 사람의 결정 → 도구 실행 또는 대체 응답만 분리해서 보여 줍니다. shopagent에는 담당자 연결, 권한·정책 확인, 호출 횟수 제한 등의 추가 흐름이 있습니다.

## 4. 직접 바꿔 볼 것

먼저 가짜 모델로 해 보세요. 아래 가짜 모델 결과는 검증한 동작입니다. 실모델이 어느 도구를 선택할지는 여기서 검증하지 않았습니다. 명령 예시는 새 코드 셀에서 실행하면 됩니다.

1. **같은 대화와 새 대화:** 1부의 두 번째 호출에서 `config_a`를 그대로 쓸 때와 `thread_config("다른대화")`로 바꿀 때를 비교하세요. 처음부터 실행하면 저장된 메시지는 각각 4개와 2개입니다.
2. **저장소 교체와 커널 재시작:** `fresh_graph = memory_builder.compile(checkpointer=InMemorySaver())` 다음 `fresh_graph.get_state(config_a).values`를 보세요. 같은 이름표여도 빈 상태입니다. 기존 그래프와 별도 저장소이기 때문입니다. 커널 재시작은 파이썬 객체 자체를 없애므로 기존 메모리 기록도 없어집니다. 재시작 실험은 다른 실험을 마친 뒤 하세요.
3. **배송중 주문:** `cfg, pending, _ = start_cancel_case("shipped", text="주문 O003 취소해 주세요.")` 후 `resume_case(order_agent, cfg, pending, {"type": "approve"})`를 실행하세요. 승인은 실행 허가일 뿐입니다. 취소 함수는 실행되지만 배송중이라 거부하고 주문은 그대로입니다. `ToolMessage.status`가 success여도 내용의 `ok`는 False입니다.
4. **실제 모델의 도구 선택:** 서버나 키를 준비한 뒤 `USE`를 gemma 또는 solar로 바꾸고 선택 셀부터 다시 실행하세요. 취소 요청에서 바로 도구 호출과 중단이 나오는지, 글로만 확인하는지 관찰하세요. Solar의 실제 호출은 유료입니다. 이 노트북의 가짜 모델 검증에서는 실모델을 호출하지 않았습니다.
5. **허용하지 않은 결정:** `strict_agent = make_order_agent(allowed_decisions=["approve"])`로 만들고 새 취소 중단을 받으세요. 여기에 reject를 넣어 재개하면 입력 검증 오류가 납니다. 주문은 그대로이고 중단은 남습니다. 같은 중단 ID에 approve를 다시 넣으면 재개됩니다. 오류 확인은 `try/except`로 감싸면 다음 실험도 이어서 할 수 있습니다.
6. **조회도 승인받기:** `read_agent = make_order_agent(interrupt_reads=True)`로 만든 뒤 새 대화에서 O001 조회를 요청하세요. 이제 get_order 실행 전에도 중단됩니다. approve로 재개하면 조회가 실행되며 주문은 바뀌지 않습니다.
7. **거절 후 다시 요청:** 2부의 거절 셀 직후 같은 `reject_config`에 `{"messages": [HumanMessage(content="주문 O001 취소해 주세요.")]}`를 다시 보내세요. 가짜 모델은 다시 도구 호출을 내고 새 중단이 생깁니다. 아직 도구는 실행되지 않습니다. 이것은 이 노트북의 동작입니다. shopagent에는 위 표의 별도 거절 이력 정책이 있습니다.

실험 도우미 `start_cancel_case()`는 주문을 매번 초기화합니다. 기존 중단을 남겨 둔 채 다른 예시를 실행하면 주문 자료는 공유되므로, 한 실험의 중단과 재개를 먼저 끝내고 다음 실험으로 넘어가세요.